# Étiquetage automatique de questions posées sur StackOverflow

Projet n$^\text{o}$ 4 du [cursus Machine Learning Engineer][2] d'OpenClassrooms

Auteur : [Kiril ISAKOV][1]

Mentor : Nicolas TISSERAND

Projet démarré le 02/06/2025

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

## Énoncé

1. Requête SQL pour récupérer les données depuis [l'API StackExchange][1] :

    ```sql
    SELECT TOP 50000
      Title
      , Body
      , Tags
      , Score
      , AnswerCount
    FROM
      Posts
    WHERE
      PostTypeId = 1
      AND LEN(Tags) >= 1
      AND YEAR(CreationDate) = 2024
      AND MONTH(CreationDate) = 1 --1, 2, ..., 12
    ```

[1]: https://data.stackexchange.com/stackoverflow/query

## Étapes préliminaires

### Imports et constantes

In [35]:
from bs4 import BeautifulSoup
from collections import Counter
from collections.abc import Callable
from functools import wraps
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
# from google.colab import drive
from io import StringIO
from pathlib import Path, PosixPath
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.exceptions import ConvergenceWarning, InconsistentVersionWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# FutureWarning: Class PassiveAggressiveClassifier is deprecated in version 1.8 and will be removed in 1.10. Use
# `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier, SGDClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.utils._testing import ignore_warnings
from tqdm.notebook import tqdm, trange
from typing import List, Union, Any
import ast
import datetime as dt
import gc
import hashlib
import itertools as it
import joblib
import logging
import math
import mlflow
import mlflow.sklearn
import multiprocessing as mp
import nltk
import numpy as np
import os
import pandas as pd
import pickle
import pytz
import re
import seaborn as sns
import sys
import time
import torch
import transformers
import wandb
import warnings

# nltk.download('omw-1.4')
# nltk.download('punkt_tab')
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

sns.set()

PROJECT_DIR = '2025.06.02.OC.proj.04.NLP.StackOverflow.tags'
DB_FILENAME_GLOB = 'so.2024.*.csv'
VECTORIZER_TOKEN_REGEX_PATTERN = r'\b[a-zA-Z0-9.][a-zA-Z0-9_+#~.-]*[a-zA-Z0-9+#]'
TAGS_SEPARATOR_REGEX_PATTERN = re.compile(r'[<>]')
WANDB_API_KEY = '4c856baf4dc7314db3f885f12d52e9bf6873d55f'
LOGGER_FORMAT = '%(asctime)s [%(levelname)s] %(message)s'

logging.Formatter.converter = lambda *_: dt.datetime.now(pytz.timezone('Europe/Paris')).timetuple()
logging.basicConfig(level=logging.INFO, format=LOGGER_FORMAT, force=True)
logr = logging.getLogger(__name__)
logr.setLevel(logging.INFO)
log_file_handler = logging.FileHandler(f"{dt.date.isoformat(dt.datetime.now(pytz.timezone('Europe/Paris')))}.log")
log_file_handler.setLevel(logging.DEBUG)
log_cons_handler = logging.StreamHandler()
log_cons_handler.setLevel(logging.ERROR)
log_file_handler.setFormatter(LOGGER_FORMAT)
log_cons_handler.setFormatter(LOGGER_FORMAT)
logr.addHandler(log_file_handler)
logr.addHandler(log_cons_handler)

logging.getLogger('alembic').setLevel(logging.WARNING)

### Fonctions

In [ ]:
def connect_drive(drive_path: str ='/content/mnt/MyDrive',
                  nb_link_path: str ='/content/notebooks'):
    if not os.path.ismount('/content/mnt'):
        drive.mount('/content/mnt')
    if not os.path.lexists(nb_link_path):
        os.symlink(src=drive_path + '/Colab Notebooks', dst=nb_link_path)
        sys.path.insert(0, nb_link_path)

def read_nth_line(path: Union[str, Path], n: Union[int, List[int]]=1) -> str:
    """Read `n`th line or a `[n, n']` range of lines from the `path` to a text file.
    First line is both n=1 and n=0."""
    frm, to = n if isinstance(n, list) else (max(n, 1), max(n, 1))
    ret = ''
    with open(path, 'rt') as txt_rdr:
        for i, line in enumerate(txt_rdr, 1):
            if i in range(frm, to + 1):
                ret += line
    return ret

def read_header(path: Union[str, Path]) -> str:
    return read_nth_line(path)

def strip_tags_of_a_chunk(chunk: pd.DataFrame):
    logr = globals().get('logr')
    logr.info('Processing one of the chunks...')
    for i in tqdm(chunk.index, desc="Processing the rows of one of the chunks", unit='row', leave=False):
        chunk.at[i, 'tags'] = tuple(tag for tag in TAGS_SEPARATOR_REGEX_PATTERN.split(chunk.at[i, 'tags']) if tag != '')
    return chunk

def concat_body_and_title_of_a_chunk(chunk: pd.DataFrame):
    logr = globals().get('logr')
    logr.info('Processing one of the chunks...')
    for i in tqdm(chunk.index, desc="Processing the rows of one of the chunks", unit='row', leave=False):
        chunk.at[i, 'Body'] = BeautifulSoup(chunk.at[i, 'Body'], 'html.parser').get_text(strip=True).replace('\n', ' ')
        chunk.at[i, 'Body'] = chunk.at[i, 'Title'] + ' ' + chunk.at[i, 'Body']
    return chunk

def imap_unordered_worker(chunks: List[pd.DataFrame], func, path: Path, logr: logging.Logger):
    num_cores = mp.cpu_count()
    with mp.Pool(num_cores) as pool:
        logr.info(f"Processing {path.name!r} by chunks...")
        for result in pool.imap_unordered(func, chunks):
            logr.info(f"Yield one of the processed chunks")
            yield result

def strip_tags_with_multiprocessing(path: Path, logr: logging.Logger, save_result: bool = True):
    """À l'aide de la parallélisation et grâce au module `multiprocessing`,
    - Enlever les chevrons '<' et '>' qui entourent chaque tag. """
    df = pd.read_csv(path.name)
    logr.info(f"Loaded {len(df)} rows from {path.name!r}")
    num_cores = mp.cpu_count()
    warnings.filterwarnings('ignore')
    chunks = np.array_split(df, num_cores)
    logr.info(f"{path.name!r} contents has been split into {len(chunks)} chunks to be processed by {num_cores} available CPU cores")
    chunks_processed = imap_unordered_worker(
            chunks, strip_tags_of_a_chunk, path, logr)
    df = pd.concat([*chunks_processed])
    warnings.filterwarnings('default')
    logr.info(f"Processed chunks have been concatenated")
    df = df.sort_index()
    logr.info(f"Dataset has been sorted by index")
    if save_result:
        df.to_csv(path.name, index=False)
        logr.info(f"{len(df)} rows saved as {path.name!r}")

def concat_body_and_title_with_multiprocessing(path: Path, logr: logging.Logger, save_result: bool = True):
    """À l'aide de la parallélisation et grâce au module `multiprocessing`,
    - Faire fusionner la colonne 'Title' avec 'Body'.
    - Supprimer la colonne 'Title'. """
    df = pd.read_csv(path.name)
    logr.info(f"Loaded {len(df)} rows from {path.name!r}")
    num_cores = mp.cpu_count()
    warnings.filterwarnings('ignore')
    chunks = np.array_split(df, num_cores)
    logr.info(f"{path.name!r} contents has been split into {len(chunks)} chunks to be processed by {num_cores} available CPU cores")
    chunks_processed = imap_unordered_worker(
            chunks, concat_body_and_title_of_a_chunk, path, logr)
    df = pd.concat([*chunks_processed])
    warnings.filterwarnings('default')
    logr.info(f"Processed chunks have been concatenated")
    df = df.sort_index()
    logr.info(f"Dataset has been sorted by index")
    cols_to_drop = ['Title']
    df = df.drop(columns=cols_to_drop)
    logr.info(f"Cols {cols_to_drop!r} have been dropped")
    df.columns = [c.lower() for c in df.columns]
    logr.info(f"Cols have been renamed to lowercase")
    if save_result:
        df.to_csv(path.name, index=False)
        logr.info(f"{len(df)} rows saved as {path.name!r}")

def compute_lda(csr_matr, *, min_topics, max_topics, step, perplexity_threshold):
    """- Train and evaluate LDA model's perplexity (the lower the better) for
    various topic sets of sizes in `range(min_topics, max_topics + 1, step)` over a
    sparse matrix which is a product of a `CountVectorizer.fit_transform(text_documents)`
    computation.
       - If perplexity < perplexity_threshold, then save (serialize) LDA model
    object as a pickle file."""
    for n_topics in trange(min_topics, max_topics + 1, step):
        lda = LatentDirichletAllocation(n_topics, random_state=42, n_jobs=-1)
        logr.info(f"Training LDA with {n_topics} topics...")
        lda.fit(csr_matr)
        perplexity = lda.perplexity(csr_matr)
        logr.info(f"Model with {n_topics} topics -> Perplexity: {perplexity:_.2f} (the lower the better)")
        if perplexity < perplexity_threshold:
            perplexity_threshold = perplexity
            fname = f'lda_{len(feature_names)}_words_{n_topics}_topics_perplexity_{perplexity:.2f}.pkl'
            joblib.dump(lda, fname)

def get_topic_top_words(lda_model, feature_names, n_top_words=10):
    """Return list of top words per topic as a dict {topic_id: [words...]}."""
    topic_keywords = {}
    for topic_idx, topic in enumerate(lda_model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_features_ind]
        topic_keywords[topic_idx] = top_words
    return topic_keywords

@ignore_warnings(category=InconsistentVersionWarning)
def topics_tag_documents(lda_model, csr_matr, feature_names, topic_threshold=0.2, top_words=5):
    """Assign 'predicted_tags' to each document based on topic distribution.
    ⚠️ Max tags = 1 / topic_threshold × top_words ;
        e. g. 1 / 0.2 × 5 = 25"""
    # Topic-word dictionary
    topic_keywords = get_topic_top_words(lda_model, feature_names, n_top_words=top_words)

    # Document-topic distribution
    doc_topic_dist = lda_model.transform(csr_matr)

    tags_per_doc = []
    for _, topic_probs in enumerate(doc_topic_dist):
        tags = []
        for topic_idx, prob in enumerate(topic_probs):
            if prob >= topic_threshold:  # topic is relevant
                tags.extend(topic_keywords[topic_idx])
        # remove duplicates, keep order
        tags_per_doc.append(list(dict.fromkeys(tags)))
    return tags_per_doc

def extract_tags(col: pd.Series,
                 separator_pattern: re.Pattern = TAGS_SEPARATOR_REGEX_PATTERN
                 ):
    for tags in col:
        yield tuple(tag for tag in separator_pattern.split(tags) if tag != '')

def get_variable_sizes():
    variables_info = []
    for name, obj in globals().items():
        if not name.startswith('_'):
            size_bytes = sys.getsizeof(obj)

            # For DataFrames, get more accurate memory usage
            if hasattr(obj, 'memory_usage'):
                size_bytes = obj.memory_usage(deep=True).sum()

            variables_info.append({
                'Variable': name,
                'Type': type(obj).__name__,
                'Size (bytes)': size_bytes,
                'Size (MB)': size_bytes / (1024 * 1024)
            })

    df = pd.DataFrame(variables_info)
    return df.sort_values('Size (bytes)', ascending=False)

def filtre_by_top_k_tags(df_sample: pd.DataFrame,
                         top_k_tags: tuple[str],
                         sample_size: int) -> pd.DataFrame:
    logr.info("Copy the original DF so it is not altered.")
    df_sample = df.copy()
    logr.info("Filter each question's tags by keeping only those in the top_k_tags list. "
              "Drop questions with no tags left.")
    df_sample.loc[:, 'tags'] = df_sample['tags'].apply(lambda tags_tuple: tuple(tag for tag in tags_tuple if tag in top_k_tags))
    df_sample = df_sample.loc[df_sample['tags'].apply(len) > 0]
    if len(df_sample) > sample_size:
        logr.info(f"{len(df_sample) = :,} > {sample_size = :,} \N{Rightwards Double Arrow} Reduce sample size down to {sample_size = :,}")
        df_sample = df_sample.sample(sample_size, random_state=42)
    return df_sample.reset_index(drop=True)

@ignore_warnings(category=ConvergenceWarning)
def trn_and_log_mdl(X_tr_preprocessed: Union[pd.DataFrame, pd.Series] = None,
                    y_tr: pd.Series = None,
                    model_recipe: Any = None,
                    grid_params: dict = None,
                    experiment_name: str = None,
                    project_name: str = None,
                    ):
    for parm, parm_name in zip((grid_params, y_tr), ('grid_params', 'y_tr')):
        if parm is None:
            raise ValueError(f"{parm_name} must be provided")

    mlflow.set_experiment(experiment_name)
    run = wandb.init(name=experiment_name, project=project_name) if project_name else wandb.init(name=experiment_name)

    with mlflow.start_run():
        grid = GridSearchCV(
            model_recipe,
            grid_params,
            return_train_score=True,
            scoring='f1_macro',
            cv=5,
            n_jobs=-1,
            verbose=3,
        )
        gc.collect()
        grid.fit(X_tr_preprocessed, y_tr)
        display(pd.DataFrame(grid.cv_results_))
        best_index = np.where(grid.cv_results_['rank_test_score'] == 1)[0][0]
        mlflow.log_metric('mean_train_score', grid.cv_results_['mean_train_score'][best_index])
        run.log({'mean_train_score': grid.cv_results_['mean_train_score'][best_index]})
        mlflow.log_metric('mean_test_score', grid.cv_results_['mean_test_score'][best_index])
        run.log({'mean_test_score': grid.cv_results_['mean_test_score'][best_index]})
        mlflow.log_metric('mean_score_time', grid.cv_results_['mean_score_time'][best_index])
        run.log({'mean_score_time': grid.cv_results_['mean_score_time'][best_index]})
        mlflow.log_metric('mean_fit_time', grid.cv_results_['mean_fit_time'][best_index])
        run.log({'mean_fit_time': grid.cv_results_['mean_fit_time'][best_index]})
        mlflow.log_params(grid.best_params_)
        run.config.update(grid.best_params_)
        mlflow.sklearn.log_model(
            grid.best_estimator_,
            name=f'best_estimator_{experiment_name}',
            input_example=X_tr_preprocessed[:5]
        )
        joblib.dump(
            grid.best_estimator_, f'best_estimator_{experiment_name}.joblib'
        )

    run.finish()


### Charger les données

In [4]:
df = None
for path in tqdm(sorted(Path().glob(DB_FILENAME_GLOB)), leave=True, unit='file'):
    df_from_path = pd.read_csv(path.absolute(), converters={'tags': ast.literal_eval})
    df = pd.concat([df, df_from_path])
df = df.reset_index(drop=True)
df

  0%|          | 0/12 [00:00<?, ?file/s]

,body,tags,score,answercount
0,Make URL Address Text a Clickable URL Using se...,"(google-apps-script, pdf-generation)",2,1
1,7zip commandline extract excluding single file...,"(extract, 7zip)",0,1
2,How do I delete an element from an array and s...,"(javascript, arrays, google-chrome-extension, ...",0,0
3,Issues with JumpList in C# I am working on a C...,"(c#, jump-list)",0,0
4,Determine whether app launches AOT binary (or ...,"(android,)",1,0
...,...,...,...,...
443194,How to add a nullable json field to a struct a...,"(rust, yaml, jsonb, rust-cargo, rust-diesel)",0,1
443195,Tawk.to Widget Stops Working After Livewire Co...,"(javascript, laravel, laravel-livewire, livewi...",0,0
443196,Why is there a 20MB chunks folder when I switc...,"(cloudflare, filesize, astrojs, cloudflare-wor...",0,1
443197,SSIS Package Completes all Tasks but encounter...,"(sql-server, ssis, timeout)",0,0


### NLTK : Lemmatization & Stemming (🇫🇷 lemmatisation et racinisation)

Conclusions :

- Le stemming (la racinisation) n'est clairement pas pertinent.

- La lemmatisation n'est finalement pas très pertinente non plus étant donné la particularité du vocabulaire, de la syntaxe et de la ponctuation spécifiques aux langages de programmation où nombre de mots et d'usages n'existent pas dans l'anglais standard (du `snake_case`, `UPPER_SNAKE_CASE`, `camelCase`, `PascalCase`, `kebab-case`, etc).

- Les stopwords devraient très clairement être personnalisés afin d'éviter de laisser passer à la trappe des stopwords autrement banales mais qui sont pourtant chargés de sens étant donné la spécificité du champ lexical de la programmation (de faux stopwords comme `for`, `if`, `else`, `do`, `while`, `in`, `any`, `not`, etc).

### Approche non-supervisée : LDA (Latent Dirichlet allocation)

- https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html

- https://www.baeldung.com/cs/topic-modeling-coherence-score

- https://www.linkedin.com/advice/1/how-do-you-evaluate-coherence-perplexity

Recherche du nombre optimal de topics :
- algorithme utilisé : `sklearn.decomposition.LatentDirichletAllocation` ;
- métrique de performance : perplexity.

Conclusions :

- pas scalable : trop d'effort et de temps humain et de calcul pour un résultat insatisfaisant ;

- métrique possible pour évaluer quantitivement la pertinence des topics vs tags données (équivalente à une évaluation par un expert métier) :

    - à supposer que les tags de base sont (a) disponibles à la consultation, (b) fiables et (c) *idéalement* figurant parmi les mots présents dans chaque document (⚠️ ce qui n'est pas forcément le cas en réalité) :
  
        - évaluer Precision et Recall entre les topic et les tags pour chacun des doc ;

        - évaluer la similarité sémantique entre les topic et les tags pour chacun des docs

    - à supposer que les tags de base ne sont ni disponibles à la consultation ni fiables :

        - évaluer (noter) à la main (càd par un groupe d'experts) les topics prédits mais aussi les tags de base ; comparer les notes ;

        - tester la stabilité de l'approche même en entraînant, pour chaque document, plusieurs fois un LDA différent selon son paramètre `random_state`

### Approche supervisée : objectifs :

- filtrage des tags (top 100 par exemple)

- split du dataframe train/test

- extraction des features via le bag of word -> X_train ; traitement des tags -> y_train

- utilisation d'un modèle (régression logistique par exemple), plongé dans un OneVsRestClassifier

- grille de validation pour le paramètre de régularisation C (attention, ajouter estimator__ devant pour le OneVsRest)

- évaluation avec une métrique bien choisie (f1_macro ou f1_micro par exemple, puis réfléchir aux métriques)

#### Top tags

In [5]:
bag_of_tags = Counter(it.chain.from_iterable(df['tags']))
bag_of_tags_sorted_desc = dict(sorted(bag_of_tags.items(), reverse=True, key=lambda item: item[1]))
print(f'Top tags, sorted descending:\n')
display(
    dict(list(bag_of_tags_sorted_desc.items())[:10]),
)
print('  ..., etc.')

Top tags, sorted descending:



{'python': 56005,
 'javascript': 32089,
 'c#': 22260,
 'java': 19789,
 'reactjs': 18801,
 'android': 16440,
 'c++': 13009,
 'html': 12998,
 'flutter': 12756,
 'r': 12685}

  ..., etc.


#### Filtrer le DF sur les top k premiers tags

In [6]:
TOP_K = 100
SAMPLE_SIZE = 100_000
top_k_tags = tuple(bag_of_tags_sorted_desc.keys())[:TOP_K]
df_sample = filtre_by_top_k_tags(df, top_k_tags, SAMPLE_SIZE)

2026-02-16 09:20:49,580 [INFO] Copy the original DF so it is not altered.
2026-02-16 09:20:49,598 [INFO] Filter each question's tags by keeping only those in the top_k_tags list. Drop questions with no tags left.
2026-02-16 09:20:51,520 [INFO] len(df_sample) = 357,500 > sample_size = 100,000 ⇒ Reduce sample size down to sample_size = 100,000


train_test_split():

In [22]:
from sklearn.model_selection import train_test_split

logr.info("Instancier un MultiLabelBinarizer()")
mlb = MultiLabelBinarizer()
logr.info("Appliquer fit_transform() sur le champ 'tags' de notre sample")
Y = mlb.fit_transform(df_sample['tags'])
logr.info("Sauvegarder le MultiLabelBinarizer transformé en tant que pickle")
joblib.dump(mlb, f'mlb_{SAMPLE_SIZE}.pkl')
logr.info("Appliquer train_test_split() sur notre sample")
X = df_sample['body']
X_tr, X_te, y_tr, y_te = train_test_split(X, Y, test_size=0.2, random_state=42)

2026-02-16 09:57:06,140 [INFO] Instancier un MultiLabelBinarizer()
2026-02-16 09:57:06,141 [INFO] Appliquer fit_transform() sur le champ 'tags' de notre sample
2026-02-16 09:57:06,240 [INFO] Sauvegarder le MultiLabelBinarizer transformé en tant que pickle
2026-02-16 09:57:06,242 [INFO] Appliquer train_test_split() sur notre sample


#### La partie vectorizers (en choisir un et l'exécuter)

CountVectorizer (`X_tr_vec_CountVectorizer.pkl` et `vectorizer_CountVectorizer.pkl`)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

fname = f'X_tr_vec_CountVectorizer_{SAMPLE_SIZE}.pkl'
if os.path.exists(fname):
    logr.info(f"Désérialiser le pickle sauvegardé {fname!r}")
    X_tr_vec = joblib.load(fname)
else:
    vectorizer_params = dict(min_df=50, max_df=0.2, lowercase=True, token_pattern=VECTORIZER_TOKEN_REGEX_PATTERN)
    logr.info("Instancier un CountVectorizer() avec des params")
    vectorizer = CountVectorizer(**vectorizer_params)
    logr.info("Appliquer fit_transform() sur l'échantillon d'entraînement")
    X_tr_vec = vectorizer.fit_transform(X_tr)
    logr.info(f"Sauvegarder les données d'entraînement vectorisées en tant que pickle {fname!r}")
    joblib.dump(X_tr_vec, fname)
    logr.info(f"Sauvegarder le vectorizer en tant que pickle {fname.replace('X_tr_vec_', '')!r}")
    joblib.dump(vectorizer, fname.replace('X_tr_vec_', 'vectorizer_'))

2026-01-08 15:20:51,517 [INFO] Désérialiser le pickle sauvegardé 'X_tr_vec_CountVectorizer_100000.pkl'


TfidfVectorizer (`X_tr_vec_TfidfVectorizer.pkl` et `vectorizer_TfidfVectorizer.pkl`)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

fname = f'X_tr_vec_TfidfVectorizer_{SAMPLE_SIZE}.pkl'
if os.path.exists(fname):
    logr.info(f"Désérialiser le pickle sauvegardé {fname!r}")
    X_tr_vec = joblib.load(fname)
else:
    vectorizer_params = dict(min_df=50, max_df=0.2, lowercase=True, token_pattern=VECTORIZER_TOKEN_REGEX_PATTERN)
    logr.info("Instancier un TfidfVectorizer() avec des params")
    vectorizer = TfidfVectorizer(**vectorizer_params)
    logr.info("Appliquer fit_transform() sur l'échantillon d'entraînement")
    X_tr_vec = vectorizer.fit_transform(X_tr)
    logr.info(f"Sauvegarder les données d'entraînement vectorisées en tant que pickle {fname!r}")
    joblib.dump(X_tr_vec, fname)
    logr.info(f"Sauvegarder le vectorizer en tant que pickle {fname.replace('X_tr_vec_', '')!r}")
    joblib.dump(vectorizer, fname.replace('X_tr_vec_', 'vectorizer_'))

2026-01-07 07:11:03,816 [INFO] Désérialiser le pickle sauvegardé 'X_tr_vec_TfidfVectorizer_100000.pkl'


Word2Vec (`X_tr_vec_Word2Vec.pkl`)

In [ ]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

logging.getLogger('gensim').setLevel(logging.WARNING)

fname = f'X_tr_vec_Word2Vec_{SAMPLE_SIZE}.pkl'
if os.path.exists(fname):
    logr.info(f"Désérialiser le pickle sauvegardé {fname!r}")
    X_tr_vec = joblib.load(fname)
else:
    vectorizer_params = dict(min_count=50, sample=0.001, vector_size=100, seed=42, epochs=5, workers=mp.cpu_count())
    logr.info(f"Word2Vec: Initialize vectorizer with params {vectorizer_params}")
    w2v_model = Word2Vec(**vectorizer_params)
    logr.info(f"Tokenize each one of the {len(X_tr)} documents as per the same regex pattern we used earlier")
    token_pattern = re.compile(VECTORIZER_TOKEN_REGEX_PATTERN)
    tokenized = [token_pattern.findall(doc.lower()) for doc in X_tr]
    logr.info("Word2Vec: build_vocab()")
    w2v_model.build_vocab(corpus_iterable=tokenized)
    logr.info("Word2Vec: train()")
    w2v_model.train(corpus_iterable=tokenized, total_examples=w2v_model.corpus_count, epochs=5)
    logr.info(f"Word2Vec training finished. Vocabulary size: {len(w2v_model.wv.index_to_key):,}")

    logr.info("Iterate over the tokenized docs... Only keep valid tokens..."
              " Extract vectors, either as means (if valid), or as zeros (if none)...")
    docs_as_vectors = []
    for doc_tokens in tokenized:
        valid_tokens = [t for t in doc_tokens if t in w2v_model.wv]
        docs_as_vectors.append(
            np.zeros(w2v_model.vector_size) if not valid_tokens else np.mean(w2v_model.wv[valid_tokens], axis=0)
        )
    logr.info("Convert the extracted vectors to numpy array")
    X_tr_vec = np.array(docs_as_vectors)
    logr.info(f"Sauvegarder les données d'entraînement vectorisées en tant que pickle {fname!r}")
    joblib.dump(X_tr_vec, fname)

2026-01-07 08:03:08,028 [INFO] Désérialiser le pickle sauvegardé 'X_tr_vec_Word2Vec_100000.pkl'


BERT (`X_tr_vec_Bert.pkl`)

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np

fname = f'X_tr_vec_Bert_{SAMPLE_SIZE}.pkl'
if os.path.exists(fname):
    logr.info(f"Désérialiser le pickle sauvegardé {fname!r}")
    X_tr_vec = joblib.load(fname)
else:
    bert_pretrained_model_name = 'bert-base-uncased'
    logr.info(f"Instantiate a tokenizer based on {bert_pretrained_model_name}")
    tokenizer = BertTokenizer.from_pretrained(bert_pretrained_model_name)
    logr.info(f"Instantiate a model based on {bert_pretrained_model_name}")
    model = BertModel.from_pretrained(bert_pretrained_model_name)
    gpu_preferred = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logr.info(f"Set CUDA as preferred computing kernel")
    model.to(gpu_preferred)
    batch_size = 300
    logr.info(f"Iterate over the docs by batches of {batch_size}...Tokenize docs... Encode docs as vectors...")
    embeddings_list = []
    for i in trange(0, len(X_tr), batch_size):
        from_, _till = i, i + batch_size
        batch = X_tr[from_:_till].tolist()
        tokenized = tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=512)
        tokenized = {k: v.to(gpu_preferred) for k, v in tokenized.items()}
        with torch.no_grad():
            encoded = model(**tokenized)
            batch_embeddings = encoded.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings_list.append(batch_embeddings)
    logr.info(f"Stack embeddings in sequence vertically...")
    X_tr_vec = np.vstack(embeddings_list)
    logr.info(f"Sauvegarder les données d'entraînement vectorisées en tant que pickle {fname!r}")
    joblib.dump(X_tr_vec, fname)

2026-01-19 09:15:21,753 [INFO] Instantiate a tokenizer based on bert-base-uncased
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

2026-01-19 09:15:23,893 [INFO] Instantiate a model based on bert-base-uncased


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

2026-01-19 09:15:29,533 [INFO] Set CUDA as preferred computing kernel
2026-01-19 09:15:30,171 [INFO] Iterate over the docs by batches of 300...Tokenize docs... Encode docs as vectors...


  0%|          | 0/3 [00:00<?, ?it/s]

2026-01-19 09:16:05,884 [INFO] Stack embeddings in sequence vertically...
2026-01-19 09:16:05,887 [INFO] Sauvegarder les données d'entraînement vectorisées en tant que pickle 'X_tr_vec_Bert_1000.pkl'


USE (`X_tr_vec_Use.pkl`)

In [ ]:
import tensorflow_hub
import numpy as np
import gc

fname = f'X_tr_vec_Use_{SAMPLE_SIZE}.pkl'
if os.path.exists(fname):
    logr.info(f"Désérialiser le pickle sauvegardé {fname!r}")
    X_tr_vec = joblib.load(fname)
else:
    logr.info(f"Download a USE Vectorizer")
    use_vectorizer = tensorflow_hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
    batch_size = 100
    logr.info(f"Iterate over the docs by batches of {batch_size}... Encode docs as vectors...")
    embeddings_list = []
    for i in trange(0, len(X_tr), batch_size):
        batch = X_tr[i:i+batch_size].tolist()
        batch_embeddings = use_vectorizer(batch).numpy()
        embeddings_list.append(batch_embeddings)
        if i % (batch_size * 10) == 0:
            gc.collect()
    logr.info(f"Stack embeddings in sequence vertically...")
    X_tr_vec = np.vstack(embeddings_list)
    logr.info(f"Sauvegarder les données d'entraînement vectorisées en tant que pickle {fname!r}")
    joblib.dump(X_tr_vec, fname)
    del embeddings_list
    gc.collect()

2026-01-26 09:06:24,131 [INFO] Download a USE Vectorizer
2026-01-26 09:06:24,132 [INFO] Using /tmp/tfhub_modules to cache modules.
2026-01-26 09:06:24,135 [INFO] Downloading TF-Hub Module 'https://tfhub.dev/google/universal-sentence-encoder/4'.
2026-01-26 09:06:41,090 [INFO] Downloading https://tfhub.dev/google/universal-sentence-encoder/4: 337.84MB
2026-01-26 09:06:56,150 [INFO] Downloading https://tfhub.dev/google/universal-sentence-encoder/4: 697.84MB
2026-01-26 09:07:08,440 [INFO] Downloaded https://tfhub.dev/google/universal-sentence-encoder/4, Total size: 987.47MB
2026-01-26 09:07:08,441 [INFO] Downloaded TF-Hub Module 'https://tfhub.dev/google/universal-sentence-encoder/4'.
2026-01-26 09:07:17,181 [INFO] Fingerprint not found. Saved model loading will continue.
2026-01-26 09:07:17,184 [INFO] path_and_singleprint metric could not be logged. Saved model loading will continue.
2026-01-26 09:07:17,187 [INFO] Iterate over the docs by batches of 100... Encode docs as vectors...


  0%|          | 0/8 [00:00<?, ?it/s]

2026-01-26 09:07:27,974 [INFO] Stack embeddings in sequence vertically...
2026-01-26 09:07:27,978 [INFO] Sauvegarder les données d'entraînement vectorisées en tant que pickle 'X_tr_vec_Use_1000.pkl'


#### La partie MLOps

Training (après avoir choisi un vectorizer) :

In [10]:
import mlflow
import wandb

wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/mira/.netrc
wandb: Currently logged in as: kirisakow (kirisakow-cdc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Quelques avertissements :

> * FutureWarning: Class PassiveAggressiveClassifier is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
>
> * FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. See https://scikit-learn.org/stable/model_persistence.html

In [ ]:
fnames__experiments__list = [
    (f'X_tr_vec_CountVectorizer_{SAMPLE_SIZE}.pkl', (
        SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, random_state=42),
        LogisticRegression(random_state=42),
    )),
    (f'X_tr_vec_TfidfVectorizer_{SAMPLE_SIZE}.pkl', (
        SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, random_state=42),
        LogisticRegression(random_state=42),
    )),
    (f'X_tr_vec_Word2Vec_{SAMPLE_SIZE}.pkl', (
        SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, random_state=42),
        LogisticRegression(random_state=42),
    )),
    (f'X_tr_vec_Bert_{SAMPLE_SIZE}.pkl', (
        SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, random_state=42),
        LogisticRegression(solver='saga', tol=1e-3, random_state=42),
    )),
    (f'X_tr_vec_Use_{SAMPLE_SIZE}.pkl', (
        SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, random_state=42),
        LogisticRegression(solver='saga', tol=1e-3, random_state=42),
    )),
]
estimator_param_key_name_selector = {'LogisticRegression': 'C', 'SGDClassifier': 'alpha'}
for x_tr_model_instance_fname, exper_obj_list in fnames__experiments__list:
    logr.info(f"Désérialiser le pickle sauvegardé {x_tr_model_instance_fname!r}")
    if not os.path.exists(x_tr_model_instance_fname):
        raise FileNotFoundError(x_tr_model_instance_fname)
    X_tr_vec = joblib.load(x_tr_model_instance_fname)
    base_fname = x_tr_model_instance_fname.replace('X_tr_vec_', '').replace('.pkl', '')
    for exper_obj in exper_obj_list:
        exper_name = f'{base_fname}_{type(exper_obj).__qualname__}'
        logr.info(f"Lancer l'expérience {exper_name!r}")
        param_key_name = estimator_param_key_name_selector.get(type(exper_obj).__qualname__)
        trn_and_log_mdl(
            X_tr_vec, y_tr,
            OneVsRestClassifier(exper_obj, n_jobs=1), # force n_jobs=1 as the outer func already has it at -1
            grid_params = {f'estimator__{param_key_name}': [1e-2, 1e-1, 1, 1e1, 1e2],},
            experiment_name=exper_name.replace(f'_{SAMPLE_SIZE}_', f'_{len(y_tr)}_'),
            project_name='run-train-on-laptop-32GB-ram'
        )

2026-02-08 23:50:33,141 [INFO] Désérialiser le pickle sauvegardé 'X_tr_vec_CountVectorizer_100000.pkl'
2026-02-08 23:50:33,169 [INFO] Lancer l'expérience 'CountVectorizer_100000_SGDClassifier'
2026/02/08 23:50:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/08 23:50:34 INFO mlflow.store.db.utils: Updating database tables
2026/02/08 23:50:46 INFO mlflow.tracking.fluent: Experiment with name 'CountVectorizer_100000_SGDClassifier' does not exist. Creating a new experiment.


Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV 5/5] END estimator__alpha=0.1;, score=(train=0.956, test=0.525) total time= 2.0min
[CV 4/5] END estimator__alpha=0.01;, score=(train=0.954, test=0.530) total time= 2.0min
[CV 4/5] END estimator__alpha=0.1;, score=(train=0.954, test=0.530) total time= 2.0min
[CV 3/5] END estimator__alpha=0.1;, score=(train=0.955, test=0.527) total time= 2.0min
[CV 2/5] END estimator__alpha=0.01;, score=(train=0.957, test=0.527) total time= 2.0min
[CV 2/5] END estimator__alpha=0.1;, score=(train=0.957, test=0.527) total time= 2.0min
[CV 1/5] END estimator__alpha=1;, score=(train=0.956, test=0.532) total time= 2.0min
[CV 3/5] END estimator__alpha=0.01;, score=(train=0.955, test=0.527) total time= 2.0min
[CV 1/5] END estimator__alpha=0.1;, score=(train=0.956, test=0.532) total time= 2.0min
[CV 2/5] END estimator__alpha=1;, score=(train=0.957, test=0.527) total time= 2.0min
[CV 5/5] END estimator__alpha=0.01;, score=(train=0.956, test=0.525) to

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_estimator__alpha,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,120.511580,1.919126,1.741230,0.257396,0.01,{'estimator__alpha': 0.01},0.532222,0.52667,0.527345,0.529915,...,0.528287,0.002477,1,0.956074,0.957346,0.955206,0.953532,0.955595,0.955551,0.00124
1,118.934480,1.018167,1.405574,0.182920,0.10,{'estimator__alpha': 0.1},0.532222,0.52667,0.527345,0.529915,...,0.528287,0.002477,1,0.956074,0.957346,0.955206,0.953532,0.955595,0.955551,0.00124
2,121.280523,0.979239,1.567227,0.264098,1.00,{'estimator__alpha': 1},0.532222,0.52667,0.527345,0.529915,...,0.528287,0.002477,1,0.956074,0.957346,0.955206,0.953532,0.955595,0.955551,0.00124
3,121.832191,1.398772,1.755561,0.339462,10.00,{'estimator__alpha': 10.0},0.532222,0.52667,0.527345,0.529915,...,0.528287,0.002477,1,0.956074,0.957346,0.955206,0.953532,0.955595,0.955551,0.00124
4,106.143910,28.739371,1.408556,0.554977,100.00,{'estimator__alpha': 100.0},0.532222,0.52667,0.527345,0.529915,...,0.528287,0.002477,1,0.956074,0.957346,0.955206,0.953532,0.955595,0.955551,0.00124


/home/mira/.cache/pypoetry/virtualenvs/oc-stackoverflow-api-c5RXNrAM-py3.12/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


mean_fit_time,▁
mean_score_time,▁
mean_test_score,▁
mean_train_score,▁
mean_fit_time,120.51158
mean_score_time,1.74123
mean_test_score,0.52829
mean_train_score,0.95555


...

2026-02-09 12:17:37,503 [INFO] Lancer l'expérience 'Use_100000_LogisticRegression'
2026/02/09 12:17:37 INFO mlflow.tracking.fluent: Experiment with name 'Use_100000_LogisticRegression' does not exist. Creating a new experiment.


Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV 5/5] END estimator__C=0.01;, score=(train=0.006, test=0.006) total time=19.5min
[CV 1/5] END estimator__C=0.01;, score=(train=0.006, test=0.006) total time=19.6min
[CV 2/5] END estimator__C=0.01;, score=(train=0.006, test=0.006) total time=19.7min
[CV 4/5] END estimator__C=0.01;, score=(train=0.006, test=0.006) total time=19.8min
[CV 3/5] END estimator__C=0.01;, score=(train=0.006, test=0.006) total time=19.8min
[CV 2/5] END estimator__C=0.1;, score=(train=0.163, test=0.164) total time=20.8min
[CV 3/5] END estimator__C=0.1;, score=(train=0.162, test=0.160) total time=20.9min
[CV 5/5] END estimator__C=0.1;, score=(train=0.162, test=0.158) total time=21.0min
[CV 4/5] END estimator__C=0.1;, score=(train=0.163, test=0.159) total time=21.1min
[CV 1/5] END estimator__C=0.1;, score=(train=0.162, test=0.160) total time=21.2min
[CV 1/5] END estimator__C=1;, score=(train=0.455, test=0.438) total time=22.6min
[CV 2/5] END estimator__

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_estimator__C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,1180.599296,6.804194,1.106594,0.094188,0.01,{'estimator__C': 0.01},0.005908,0.005833,0.005846,0.005917,...,0.005867,0.000038,5,0.005910,0.005920,0.005847,0.005831,0.005903,0.005882,0.000036
1,1259.151985,7.192385,1.092261,0.142125,0.10,{'estimator__C': 0.1},0.159531,0.163613,0.160324,0.159392,...,0.160161,0.001889,4,0.161866,0.163242,0.162396,0.162580,0.162412,0.162499,0.000442
2,1353.800926,3.797350,1.029584,0.105726,1.00,{'estimator__C': 1},0.437704,0.436924,0.438146,0.432938,...,0.436061,0.001985,3,0.454782,0.456845,0.455651,0.457484,0.456200,0.456192,0.000936
3,3443.451231,15.767123,0.861576,0.136997,10.00,{'estimator__C': 10.0},0.524134,0.517233,0.520008,0.512551,...,0.516433,0.005567,2,0.568266,0.572014,0.569055,0.570796,0.573097,0.570645,0.001794
4,6513.458281,303.914993,0.600825,0.043405,100.00,{'estimator__C': 100.0},0.534469,0.526769,0.527402,0.517314,...,0.525099,0.006122,1,0.620813,0.623400,0.621871,0.623478,0.623988,0.622710,0.001183


/home/mira/.cache/pypoetry/virtualenvs/oc-stackoverflow-api-c5RXNrAM-py3.12/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


mean_fit_time,▁
mean_score_time,▁
mean_test_score,▁
mean_train_score,▁
mean_fit_time,6513.45828
mean_score_time,0.60083
mean_test_score,0.5251
mean_train_score,0.62271


#### Évaluer le couple gagnant

Le couple gagnant est `TfidfVectorizer` et `LogisticRegression`

Nous allons l'évaluer sur les données de test :

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pprint import pprint

with ignore_warnings(category=InconsistentVersionWarning):
    fname = f'vectorizer_TfidfVectorizer_{SAMPLE_SIZE}.pkl'
    logr.info(f"Désérialiser le pickle sauvegardé {fname!r}")
    vectorizer = joblib.load(fname)
    fname_best_model = 'best_estimator_TfidfVectorizer_80000_LogisticRegression.joblib'
    logr.info(f"Désérialiser le pickle sauvegardé {fname_best_model!r}")
    model = joblib.load(fname_best_model)
    y_pred = model.predict(vectorizer.transform(X_te))
    logr.info(f"{f1_score(y_te, y_pred, average='macro') = }")
    logr.info('classification_report:')
    logr.info(
        '\n' + classification_report(y_te, y_pred, target_names=mlb.classes_, zero_division=np.nan)
    )


2026-02-16 11:20:24,558 [INFO] Désérialiser le pickle sauvegardé 'vectorizer_TfidfVectorizer_100000.pkl'
2026-02-16 11:20:24,578 [INFO] Désérialiser le pickle sauvegardé 'best_estimator_TfidfVectorizer_80000_LogisticRegression.joblib'
2026-02-16 11:20:27,607 [INFO] f1_score(y_te, y_pred, average='macro') = 0.5999983756696791
2026-02-16 11:20:27,608 [INFO] classification_report:
2026-02-16 11:20:27,689 [INFO] 
                         precision    recall  f1-score   support

                   .net       0.34      0.18      0.24       378
               .net-8.0       0.31      0.15      0.20        99
              .net-core       0.31      0.11      0.17        96
              algorithm       0.55      0.27      0.36        99
    amazon-web-services       0.77      0.61      0.68       351
                android       0.75      0.62      0.68       908
android-jetpack-compose       0.88      0.68      0.77       168
         android-studio       0.47      0.27      0.34        77
 

------------------------- API ---------------------------